In [55]:
import pandas as pd

df = pd.read_csv('./smith_souzai.csv')
df.head()

,Timestamp,販売タイプ,商品名,ジャンル,値段,タンパク,商品名.1,ジャンル.1,値段.1,タンパク.1
0,14/05/2026 9:30:27,per pack,リングドーナツ（チョコ）,NaN,NaN,NaN,NaN,パン,149.0,5.1
1,14/05/2026 9:32:47,per pack,食パンの耳揚げ,NaN,NaN,NaN,NaN,肉・揚げ物,138.0,7.8
2,14/05/2026 9:36:47,per pack,クリスピーベイクドチーズ（小）,NaN,NaN,NaN,NaN,乳制品,321.0,17.0
3,14/05/2026 9:39:02,per pack,クリスピーベイクドチーズ（大）,NaN,NaN,NaN,NaN,乳制品,537.0,34.0
4,14/05/2026 9:42:00,per pack,懐かしいあんドーナツ,NaN,NaN,NaN,NaN,パン,138.0,6.4


In [56]:
print(df.isnull().sum())
print(df.columns.tolist())
print(df.shape)


Timestamp     0
販売タイプ         0
商品名           0
ジャンル         70
値段           70
タンパク         70
商品名.1        86
ジャンル.1        0
値段.1          0
タンパク.1        0
dtype: int64
['Timestamp', '販売タイプ', '商品名', 'ジャンル', '値段', 'タンパク', '商品名.1', 'ジャンル.1', '値段.1', 'タンパク.1']
(86, 10)


In [57]:
df = df.drop(['ジャンル','値段','タンパク','商品名.1'],axis = 1)
df.head()

,Timestamp,販売タイプ,商品名,ジャンル.1,値段.1,タンパク.1
0,14/05/2026 9:30:27,per pack,リングドーナツ（チョコ）,パン,149.0,5.1
1,14/05/2026 9:32:47,per pack,食パンの耳揚げ,肉・揚げ物,138.0,7.8
2,14/05/2026 9:36:47,per pack,クリスピーベイクドチーズ（小）,乳制品,321.0,17.0
3,14/05/2026 9:39:02,per pack,クリスピーベイクドチーズ（大）,乳制品,537.0,34.0
4,14/05/2026 9:42:00,per pack,懐かしいあんドーナツ,パン,138.0,6.4


In [58]:
df = df.rename(columns={'ジャンル.1':'カテゴリ'})
df = df.rename(columns={'値段.1':'値段'})
df = df.rename(columns={'タンパク.1':'タンパク'})

df.head()

,Timestamp,販売タイプ,商品名,カテゴリ,値段,タンパク
0,14/05/2026 9:30:27,per pack,リングドーナツ（チョコ）,パン,149.0,5.1
1,14/05/2026 9:32:47,per pack,食パンの耳揚げ,肉・揚げ物,138.0,7.8
2,14/05/2026 9:36:47,per pack,クリスピーベイクドチーズ（小）,乳制品,321.0,17.0
3,14/05/2026 9:39:02,per pack,クリスピーベイクドチーズ（大）,乳制品,537.0,34.0
4,14/05/2026 9:42:00,per pack,懐かしいあんドーナツ,パン,138.0,6.4


In [59]:
df_pack = df[df['販売タイプ']=='per pack']
df_per100= df[df['販売タイプ']== 'per 100gr']

print(df_pack.shape)
print(df_per100.shape)

(70, 6)
(16, 6)


In [60]:
df_clean = pd.concat([df_pack,df_per100],axis=0)
df_clean.shape


(86, 6)

In [61]:
print(df_clean.columns)

Index(['Timestamp', '販売タイプ', '商品名', 'カテゴリ', '値段', 'タンパク'], dtype='object')


In [62]:
df_clean['値段'] = pd.to_numeric(df_clean['値段'],errors='coerce')
df_clean['タンパク'] = pd.to_numeric(df_clean['タンパク'],errors='coerce')
print(df_clean.dtypes)

Timestamp     object
販売タイプ         object
商品名           object
カテゴリ          object
値段           float64
タンパク         float64
dtype: object


In [64]:
#calcutaled field

df_clean['p/c_score'] = (df_clean['タンパク']/df_clean['値段'] * 100).round(2)

def pc_level(score):
    if score >= 5.0 : return 'high'
    elif score >= 2.0 : return 'mid'
    else : return 'low'
    
df_clean['pc_level'] = df_clean['p/c_score'].apply(pc_level)

df_clean.head()


,Timestamp,販売タイプ,商品名,カテゴリ,値段,タンパク,p/c_score,pc_level
0,14/05/2026 9:30:27,per pack,リングドーナツ（チョコ）,パン,149.0,5.1,3.42,mid
1,14/05/2026 9:32:47,per pack,食パンの耳揚げ,肉・揚げ物,138.0,7.8,5.65,high
2,14/05/2026 9:36:47,per pack,クリスピーベイクドチーズ（小）,乳制品,321.0,17.0,5.30,high
3,14/05/2026 9:39:02,per pack,クリスピーベイクドチーズ（大）,乳制品,537.0,34.0,6.33,high
4,14/05/2026 9:42:00,per pack,懐かしいあんドーナツ,パン,138.0,6.4,4.64,mid
